In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Initial Setup
vocab_size = 2
d_model = 3
d_ff = 4
seq_len = 2


X = torch.tensor([[0.1, 0.5, -0.2],
                 [-0.3, 0.2, 0.8]], dtype=torch.float32)
target_indices = torch.tensor([1,0])
y_one_hot = torch.eye(vocab_size)[target_indices]

torch.manual_seed(42)
W1 = nn.Parameter(torch.randn(d_model, d_ff) * 0.1)
W_vocab = nn.Parameter(torch.randn(d_ff, d_model) * 0.1)

learning_rate = 0.5

## Training Loop
for i in range(50):

    H_linear = X @ W1
    H_relu = torch.maximum(H_linear, torch.tensor(0.0))
    z = H_relu @ W_vocab
    probs = F.softmax(z, dim=-1)

    loss = F.cross_entropy(z, target_indices)

    if i % 10 == 0:
        print(f"--- Run {i} ---")
        print(f"Loss: {loss.item():.4f}")

    loss.backward() ## Pytorch magic - computes all gradients automatically

    
    with torch.no_grad(): # Don't track gradients for the update itself
        W1 -= learning_rate * W1.grad
        W_vocab -= learning_rate * W_vocab.grad

        # Clear gradients for next iteration
        W1.grad = None
        W_vocab.grad = None


--- Run 0 ---
Loss: 1.1041
--- Run 10 ---
Loss: 0.9915
--- Run 20 ---
Loss: 0.5999
--- Run 30 ---
Loss: 0.3837
--- Run 40 ---
Loss: 0.1215


In [11]:
## Below is the code of using torch optimizer and then checking final loss and computing predictions



import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Setup the device (Your Mac GPU!)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# 2. The Data (Moved to MPS device)
vocab_size = 2
d_model = 3
d_ff = 4
seq_len = 2

X = torch.tensor([[0.1, 0.5, -0.2],
                  [-0.3, 0.2, 0.8]], dtype=torch.float32, device=device)
target_indices = torch.tensor([1, 0], device=device)

# 3. The Weights (Now as nn.Parameter, which tells PyTorch to track gradients!)
torch.manual_seed(42) # For reproducibility
W1 = nn.Parameter(torch.randn(d_model, d_ff, device=device) * 0.1)
W_vocab = nn.Parameter(torch.randn(d_ff, vocab_size, device=device) * 0.1)

learning_rate = 0.5

# 4. The Optimizer (Replaces manual weight updates)
optimizer = torch.optim.SGD([W1, W_vocab], lr=learning_rate)

# 5. The Training Loop
print("\n--- Starting Training ---")
for i in range(50):
    # --- FORWARD PASS (Identical math to your NumPy code) ---
    H_linear = X @ W1
    H_relu = torch.maximum(H_linear, torch.tensor(0.0, device=device))
    z = H_relu @ W_vocab  # These are the logits
    
    # PyTorch has a highly optimized, numerically stable cross-entropy built-in.
    # Note: F.cross_entropy expects RAW LOGITS (z), not probabilities! 
    # It applies softmax and computes the loss in one fused, stable operation.
    loss = F.cross_entropy(z, target_indices)
    
    if i % 10 == 0:
        print(f"Run {i:2d} | Loss: {loss.item():.4f}")
    
    # --- BACKWARD PASS (The Magic) ---
    optimizer.zero_grad()  # 1. Clear gradients from the previous step
    loss.backward()        # 2. Compute ALL gradients automatically via Autograd
    optimizer.step()       # 3. Update W1 and W_vocab using the computed gradients

print("\n--- Training Complete ---")
print(f"Final Loss: {loss.item():.4f}")

# Verify predictions
with torch.no_grad():
    final_logits = (torch.maximum(X @ W1, torch.tensor(0.0, device=device))) @ W_vocab
    final_probs = F.softmax(final_logits, dim=-1)
    print(f"\nFinal predictions for 'a' (pos 0): {final_probs[0].cpu().numpy()} (Should be close to [0, 1])")
    print(f"Final predictions for 'b' (pos 1): {final_probs[1].cpu().numpy()} (Should be close to [1, 0])")

Using device: mps

--- Starting Training ---
Run  0 | Loss: 0.6869
Run 10 | Loss: 0.6029
Run 20 | Loss: 0.3629
Run 30 | Loss: 0.1856
Run 40 | Loss: 0.0744

--- Training Complete ---
Final Loss: 0.0393

Final predictions for 'a' (pos 0): [0.05773499 0.94226503] (Should be close to [0, 1])
Final predictions for 'b' (pos 1): [0.9855299  0.01447006] (Should be close to [1, 0])


In [14]:
class TinyNet(nn.Module):
    def __init__(self, d_model, d_ff, vocab_size):
        super().__init__()

        self.W1 = nn.Linear(d_model, d_ff, bias=False)
        self.W_vocab = nn.Linear(d_ff, vocab_size, bias=False)

    def forward(self, x):
        h = F.relu(self.W1(x))
        logits = self.W_vocab(h)
        return logits

# Move x, target_indices to device
X = torch.tensor([[0.1, 0.5, -0.2],
                  [-0.3, 0.2, 0.8]], dtype=torch.float32, device=device)
target_indices = torch.tensor([1, 0], device=device)

model = TinyNet(d_model, d_ff, vocab_size)
model = model.to(device) # Move model to device

optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

for i in range(50):
    logits = model(X)
    loss = F.cross_entropy(logits, target_indices)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if i % 10 == 0:
        print(f"Run {i}: Loss = {loss.item():.4f}")

Run 0: Loss = 0.7464
Run 10: Loss = 0.6773
Run 20: Loss = 0.5975
Run 30: Loss = 0.4591
Run 40: Loss = 0.3919
